In [2]:
import pandas as pd
import os

# Find the file automatically
csv_path = None
for root, dirs, files in os.walk('/content'):
    for file in files:
        if 'product_usage' in file.lower():
            csv_path = os.path.join(root, file)
            print(f"Found file at: {csv_path}")

# Load data
df = pd.read_csv(csv_path)
df['team'] = df['team'].str.title()
df['median_confidence'] = pd.to_numeric(df['median_confidence'], errors='coerce')
df = df.drop_duplicates()
df['suspicious'] = df['notes'].str.contains('spike|duplicate|demo', case=False)
clean_df = df[~df['suspicious']]

# Summary by workflow
summary = clean_df.groupby('workflow').agg(
    total_sessions=('sessions', 'sum'),
    total_accepted=('accepted_output', 'sum'),
    total_completed=('completed', 'sum'),
    total_flagged=('flagged_for_review', 'sum'),
    avg_minutes_saved=('avg_minutes_saved', 'mean'),
    avg_user_rating=('user_rating', 'mean')
).reset_index()

summary['acceptance_rate'] = (summary['total_accepted'] / summary['total_completed']).round(2)
summary['flag_rate'] = (summary['total_flagged'] / summary['total_completed']).round(2)

print("\n=== SignalDesk Workflow Health Check ===\n")
print(summary[['workflow','total_sessions','acceptance_rate','flag_rate','avg_minutes_saved','avg_user_rating']].to_string(index=False))

print("\n=== Key Findings ===")
print("1. Lead summary: highest acceptance rate and user ratings")
print("2. Reply draft: Aug 7 flag spike due to policy change — investigate before rollout")
print("3. Feedback clustering: saves most minutes but small sample size")
print("4. Removed: duplicate row, demo spike from Aug 5")

print("\n=== Recommendation ===")
print("Trust Lead summary most. Hold Reply draft expansion until Aug 7 anomaly is understood.")
print("Gather more Feedback clustering data before drawing conclusions.")

Found file at: /content/product_usage_events.csv

=== SignalDesk Workflow Health Check ===

           workflow  total_sessions  acceptance_rate  flag_rate  avg_minutes_saved  avg_user_rating
Feedback clustering             207             0.66       0.19          12.392857         3.657143
       Lead summary             450             0.78       0.10           7.475000         4.066667
        Reply draft             510             0.76       0.17           3.576923         3.816667

=== Key Findings ===
1. Lead summary: highest acceptance rate and user ratings
2. Reply draft: Aug 7 flag spike due to policy change — investigate before rollout
3. Feedback clustering: saves most minutes but small sample size
4. Removed: duplicate row, demo spike from Aug 5

=== Recommendation ===
Trust Lead summary most. Hold Reply draft expansion until Aug 7 anomaly is understood.
Gather more Feedback clustering data before drawing conclusions.
